In [4]:
import json
from transformers import AutoTokenizer
from tqdm import tqdm

def split_dataset():
    # Указываем пути к файлам
    input_file = "dataset/master_dataset_clean.jsonl"
    out_bucket_1 = "dataset/distill/tales_bucket1_ideal.jsonl"     # <= 1536
    out_bucket_2 = "dataset/distill/tales_bucket2_condense.jsonl"  # 1537 - 4096
    out_bucket_3 = "dataset/distill/tales_bucket3_drop.jsonl"      # > 4096

    # Загружаем токенизатор целевой модели
    print("Загрузка токенизатора...")
    tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-2.8b")

    # Списки для хранения данных
    bucket_1, bucket_2, bucket_3 = [], [], []

    print("Анализ датасета...")
    with open(input_file, "r", encoding="utf-8") as f:
        lines = f.readlines()
        
        for line in tqdm(lines, desc="Подсчет токенов"):
            if not line.strip():
                continue
                
            data = json.loads(line)
            # Предполагается, что текст сказки лежит в ключе 'text'
            text = data.get("completion", "") 
            
            # Считаем токены
            token_count = len(tokenizer.encode(text))

            if token_count <= 1536:
                bucket_1.append(data)
            elif token_count <= 4096:
                bucket_2.append(data)
            else:
                bucket_3.append(data)

    # Функция для сохранения JSONL
    def save_jsonl(data_list, filename):
        with open(filename, "w", encoding="utf-8") as f:
            for item in data_list:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

    # Сохраняем результаты
    print(f"\nСохранение результатов...")
    save_jsonl(bucket_1, out_bucket_1)
    save_jsonl(bucket_2, out_bucket_2)
    save_jsonl(bucket_3, out_bucket_3)

    print("-" * 30)
    print("Готово! Статистика разбивки:")
    print(f"Корзина 1 (Идеальные, <= 1536): {len(bucket_1)} шт.")
    print(f"Корзина 2 (На сжатие, 1537-4096): {len(bucket_2)} шт.")
    print(f"Корзина 3 (Исключены, > 4096): {len(bucket_3)} шт.")
    print("-" * 30)

if __name__ == "__main__":
    split_dataset()

Загрузка токенизатора...
Анализ датасета...


Подсчет токенов: 100%|███████████████████████████████████████████████████████████████████████████████| 4355/4355 [00:17<00:00, 253.04it/s]



Сохранение результатов...
------------------------------
Готово! Статистика разбивки:
Корзина 1 (Идеальные, <= 1536): 1936 шт.
Корзина 2 (На сжатие, 1537-4096): 1745 шт.
Корзина 3 (Исключены, > 4096): 674 шт.
------------------------------


In [8]:
import json
import time
from openai import OpenAI

# Настройки твоего локального API-сервера (по умолчанию порт LM Studio - 1234)
# Если используешь другой порт, просто измени base_url
client = OpenAI(base_url="http://localhost:1234/v1", api_key="not-needed")

SYSTEM_PROMPT = """You are an expert folklorist and a conservative editor of authentic 18th-to-19th-century folklore. Your task is to condense the provided fairy tale to fit a strict length limit while preserving its core plot and atmosphere. Follow these rules strictly:

1. NO CENSORSHIP: Do not soften or "Disneyfy" the text. Preserve all dark, macabre, or violent elements typical of original folklore.
2. LENGTH CONSTRAINT: Compress the story to be around 1000-1200 words. Remove excessive descriptions and secondary subplots, but keep the main narrative intact and logically sound.
3. TONE: Maintain an archaic, objective narrative tone. Do not add artificial moral lessons or modern empathy.

Output ONLY the condensed story text. Do not add any introductory or concluding remarks."""

def run_benchmark():
    # Читаем первую строчку из корзины на сжатие
    with open("dataset/distill/tales_bucket2_condense.jsonl", "r", encoding="utf-8") as f:
        first_line = f.readline()
        data = json.loads(first_line)
        original_text = data.get("completion", "")

    print(f"Длина оригинала: {len(original_text.split())} слов.")
    print("Отправка запроса в Qwen 32B... Ждем генерацию...\n")

    start_time = time.time()

    # Делаем вызов к локальной модели
    response = client.chat.completions.create(
        model="local-model",  # Имя игнорируется локальным сервером
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Story:\n{original_text}"}
        ],
        temperature=0.3, # Низкая температура, чтобы модель не фантазировала, а работала с текстом
        max_tokens=2000  # С запасом под наш лимит слов
    )

    end_time = time.time()
    
    result_text = response.choices[0].message.content
    duration = end_time - start_time
    
    # Считаем слова и оцениваем токены (в английском ~1.3 токена на слово)
    word_count = len(result_text.split())
    approx_tokens = word_count * 1.3
    
    print("-" * 50)
    print(f"Результат (первые 300 символов):\n{result_text[:300]}...\n")
    print("-" * 50)
    print(f"Время выполнения: {duration:.2f} секунд")
    print(f"Длина результата: {word_count} слов")
    
    if duration > 0:
        tps = approx_tokens / duration
        print(f"Примерная скорость: {tps:.1f} токенов в секунду")

if __name__ == "__main__":
    run_benchmark()

Длина оригинала: 1887 слов.
Отправка запроса в Qwen 32B... Ждем генерацию...

--------------------------------------------------
Результат (первые 300 символов):
There once lived a king and queen who were long married but childless until the queen bore a baby-boy while the king was away in distant lands. The queen refused to christen him, saying, “We will just call him Nix Nought Nothing until his father returns.” As the boy grew into a laddie, the king retu...

--------------------------------------------------
Время выполнения: 16.64 секунд
Длина результата: 442 слов
Примерная скорость: 34.5 токенов в секунду


In [13]:
import json
import time
from openai import OpenAI

# Настройки локального API-сервера
client = OpenAI(base_url="http://localhost:1234/v1", api_key="not-needed")

SYSTEM_PROMPT = """You are an expert folklorist and a conservative editor of authentic 18th-to-19th-century folklore. Your task is to rewrite and logically repair the provided fairy tale WITHOUT over-summarizing it. 

Follow these rules strictly:
1. NO CENSORSHIP: Preserve all dark, macabre, or violent elements typical of original folklore. Do not modernize the text.
2. PRESERVE DETAIL: Do NOT write a short summary. Keep all secondary characters, dialogues, and descriptive atmosphere. Your goal is a rich, full-length narrative of about 1000-1200 words.
3. LOGICAL REPAIR: Fix any broken plots or sudden jumps, but keep the exact pacing of a traditional fairy tale.
4. TONE: Maintain an archaic, objective narrative tone. 

Output ONLY the story text. Do not add any introductory or concluding remarks."""

def process_specific_tale(target_index=0):
    input_file = "dataset/distill/tales_bucket2_condense.jsonl"
    original_text = ""
    
    # Ищем нужную строку по индексу
    try:
        with open(input_file, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i == target_index:
                    data = json.loads(line)
                    original_text = data.get("completion", "")
                    break
            else:
                print(f"Ошибка: Сказка с индексом [{target_index}] не найдена. В файле меньше строк.")
                return
    except FileNotFoundError:
        print(f"Ошибка: Файл {input_file} не найден.")
        return

    print(f"--- Выбрана сказка с индексом [{target_index}] ---")
    print(f"Длина оригинала: {len(original_text.split())} слов.")
    print("Отправка запроса в Qwen 32B... Ждем генерацию...\n")

    start_time = time.time()

    response = client.chat.completions.create(
        model="local-model",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Story:\n{original_text}"}
        ],
        temperature=0.3,
        max_tokens=2048
    )

    end_time = time.time()
    
    result_text = response.choices[0].message.content
    duration = end_time - start_time
    word_count = len(result_text.split())
    approx_tokens = word_count * 1.3
    
    print("-" * 50)
    print(f"Результат \n{result_text}...\n")
    print("-" * 50)
    print(f"Время выполнения: {duration:.2f} секунд")
    print(f"Длина результата: {word_count} слов")
    
    if duration > 0:
        tps = approx_tokens / duration
        print(f"Примерная скорость: {tps:.1f} токенов/сек")

if __name__ == "__main__":
    # Укажи здесь нужный индекс сказки (0, 1, 2, 3...)
    process_specific_tale(target_index=108)

--- Выбрана сказка с индексом [108] ---
Длина оригинала: 1740 слов.
Отправка запроса в Qwen 32B... Ждем генерацию...

--------------------------------------------------
Результат 
In a village nestled amidst rolling hills and dense forests dwelt a poor widow named Margaret. She had one son, Jack Dreadnought, so bold that even tales of the devil's mother failed to instill any fear within him. His mother, Margaret, was perpetually anxious about her son’s foolhardy nature and sought every means possible to teach him to be cautious.

Margaret decided to send Jack to the village clergyman, a learned man named Reverend Thomas, with the hope that he could impart some sense of fear into the boy. The reverend tried various methods, recounting ghostly tales and sharing stories of terrifying creatures, but these efforts only piqued Jack’s curiosity rather than instilling any dread.

Seeing no other recourse, Reverend Thomas devised a plan to use dummies dressed as ghosts in an attempt to frighten